In [ ]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.data import load_unifac_data
from src.mixture.unifac.pyomo_iss import pyo_iss
from src.mixture.unifac.pyomo_iss_new import pyo_iss_new
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# step1: define the compounds
mixture = {'acetone': {'CH3':1, 'CH3CO':1},
           'methanol': {'CH3OH':1},
           'water': {'H2O':1}}
# validate the structures
validator = StructuralValidator()
validator.validate(mixture)

{'acetone': True, 'methanol': True, 'water': True}

In [3]:
# step2: calculate the required properties
prop_list = ['tc', 'pc', 'tb', 'ac']
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)
# fix the properties of acetone
pure_props['acetone']['tc'] = 508.1
pure_props['acetone']['pc'] = 48.0
pure_props['acetone']['tb'] = 329.3
pure_props

{'acetone': {'tc': 508.1, 'pc': 48.0, 'tb': 329.3, 'ac': 0.33226923286882887},
 'methanol': {'tc': 513.0000000000005,
  'pc': 80.99999999999997,
  'tb': 337.84999999999985,
  'ac': 0.5639909999999987},
 'water': {'tc': 646.9999999999999,
  'pc': 220.6400000000002,
  'tb': 373.1499999999999,
  'ac': 0.3448609999999985}}

In [4]:
boiling_points = [props['tb'] for props in pure_props.values()]
T_min = min(boiling_points) 
T_max = max(boiling_points) 

In [5]:
# step3: calculate the Antoine coefficients
antoine_params = {}
antoine_calc = AntoineCalculator()
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props)
    print('optimization results was', results['termination_condition'] )
    antoine_params[compound] = {}
    antoine_params[compound]['A'] = results['A']
    antoine_params[compound]['B'] = results['B']
    antoine_params[compound]['C'] = results['C']
antoine_params

optimization results was optimal
optimization results was optimal
optimization results was optimal


{'acetone': {'A': 5.105487152022129,
  'B': 1863.5123768521207,
  'C': 36.111024354679266},
 'methanol': {'A': 5.253881516250653,
  'B': 1616.1411202945164,
  'C': -29.905956718707568},
 'water': {'A': 5.276150790167597,
  'B': 1810.313164589222,
  'C': -29.66536409838349}}

In [6]:
# Helper function to create initialization dictionaries
def create_init_dict_1d(values, fallback):
    """Create 1D initialization dict"""
    if init_values is not None and values is not None:
        return {i: float(values[i]) for i in range(len(values))}
    return fallback

def create_init_dict_2d(values, fallback, keys1=None, keys2=None):
    """Create 2D initialization dict with optional custom keys"""
    if init_values is not None and values is not None:
        values = np.atleast_2d(values)  # Ensure 2D for single data point case
        if keys1 is not None and keys2 is not None:
            return {(k1, k2): float(values[i, j]) for i, k1 in enumerate(keys1) for j, k2 in enumerate(keys2)}
        else:
            return {(j, i): float(values[j, i]) for j in range(values.shape[0]) for i in range(values.shape[1])}
    return fallback

def create_init_dict_3d(values, fallback, keys1=None, keys2=None, keys3=None):
    """Create 3D initialization dict with optional custom keys"""
    if init_values is not None and values is not None:
        # Ensure at least 3D (handle single data point case)
        if values.ndim == 2:
            values = values[np.newaxis, :, :]  # Add data point dimension
        if keys1 is not None and keys2 is not None and keys3 is not None:
            return {(k1, k2, k3): float(values[i, j, k]) 
                    for i, k1 in enumerate(keys1) 
                    for j, k2 in enumerate(keys2) 
                    for k, k3 in enumerate(keys3)}
        else:
            return {(j, i, k): float(values[j, i, k]) for j in range(values.shape[0]) 
                    for i in range(values.shape[1]) for k in range(values.shape[2])}
    return fallback

In [7]:
#init_values = _generate_initialization_values(x0, np.array([350]), mixture, rk_data, qk_data, interaction_data, antoine_params)

In [8]:
#init_values['J']

In [9]:
#create_init_dict_2d(init_values['J'] if init_values else None, 1.13)

In [ ]:
# suggest initial values of x and T
T0 = np.array([(T_max + T_min) / 2])
xA = 0.5
x_iso = 0.75
x0 = np.array([[xA, 0.0, 1-xA]])
# load unifac data
rk_data, qk_data, interaction_data = load_unifac_data()
# perform the VLE calculation
results, model = pyo_iss_new(x0, T0, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=False)

In [11]:
# retrieving the solutions
gamma = np.empty_like(x0)
T = np.empty_like(T0)
y = np.empty_like(x0)
x = np.empty_like(x0)
psat = np.empty_like(x0)


for i in model.gamma.keys():
    gamma[i] = model.gamma[i].value

for i in model.T.keys():
    T[i] = model.T[i].value

for i in model.y.keys():
    y[i] = model.y[i].value

for i in model.x.keys():
    x[i] = model.x[i].value

for i in model.psat.keys():
    psat[i] = model.psat[i].value


print(x)
print(model.obj())

[[ 7.50000002e-01 -1.97123736e-44  2.49999998e-01]]
0.08191284375562102


In [12]:
obj_value = model.obj()
obj_value

0.08191284375562102

In [13]:
model.obj()

0.08191284375562102